# Checkpoint 4 Gold Layer, EDA, RFM Model, and Dashboard Data

**Team 9:** Arpan, Vishal, Pranav, Abinash, Rupjeet, and Anil  
**Dataset:** UCI Online Retail II, CC BY 4.0  
**Purpose:** Complete the gold layer, create dashboard-ready analytical tables, run exploratory analysis, and evaluate an RFM customer-segmentation model.

This notebook assumes that `checkpoint3_bronze_silver` has already created the silver table and the quality log. Run this notebook after the bronze and silver notebook. The notebook is designed for Databricks Free Edition, so it starts with table checks and keeps calculations on the aggregated customer, product, country, and date grains needed by the dashboard.

In [ ]:
from pyspark.sql import functions as F, types as T, Window
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler

catalog_name = "workspace"
schema_name = "default"
table_prefix = "retail_project"

silver_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver"
exceptions_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver_exceptions"
quality_table = f"{catalog_name}.{schema_name}.{table_prefix}_quality_log"

gold_daily_table = f"{catalog_name}.{schema_name}.{table_prefix}_gold_daily_country"
gold_monthly_table = f"{catalog_name}.{schema_name}.{table_prefix}_gold_monthly"
gold_country_table = f"{catalog_name}.{schema_name}.{table_prefix}_gold_country"
gold_product_table = f"{catalog_name}.{schema_name}.{table_prefix}_gold_product"
gold_customer_table = f"{catalog_name}.{schema_name}.{table_prefix}_gold_customer_rfm"
model_eval_table = f"{catalog_name}.{schema_name}.{table_prefix}_rfm_model_evaluation"
model_table = f"{catalog_name}.{schema_name}.{table_prefix}_rfm_segments"

required_tables = [silver_table, exceptions_table, quality_table]
missing_tables = [name for name in required_tables if not spark.catalog.tableExists(name)]
if missing_tables:
    raise ValueError(f"Run checkpoint3_bronze_silver first. Missing tables: {missing_tables}")

silver = spark.table(silver_table)
print(f"Silver table: {silver_table}")
print(f"Silver rows: {silver.count()}")
display(silver.limit(10))

## 1. Gold business rules

The gold layer uses clean silver rows and excludes cancelled invoices from sales totals. Customer analysis excludes missing customer IDs because those rows cannot be assigned to a customer. The tables retain the source grain needed for drill-down while exposing aggregates for reporting.

In [ ]:
sales = (
    silver
    .filter(~F.col("is_cancelled"))
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .withColumn("sale_date", F.to_date("invoice_date"))
    .withColumn("month_start", F.to_date(F.date_trunc("month", "invoice_date")))
    .withColumn("revenue", F.col("line_value").cast("double"))
    .withColumn("customer_id_num", F.col("customer_id").cast("double"))
)

daily_country = (
    sales.groupBy("sale_date", "country")
    .agg(
        F.sum("revenue").alias("revenue"),
        F.sum("quantity").alias("units"),
        F.countDistinct("invoice_id").alias("orders"),
        F.countDistinct("customer_id_num").alias("customers"),
        F.countDistinct("stock_code").alias("products")
    )
    .withColumn("average_order_value", F.when(F.col("orders") > 0, F.col("revenue") / F.col("orders")))
)

monthly = (
    sales.groupBy("month_start")
    .agg(
        F.sum("revenue").alias("revenue"),
        F.sum("quantity").alias("units"),
        F.countDistinct("invoice_id").alias("orders"),
        F.countDistinct("customer_id_num").alias("customers")
    )
    .withColumn("average_order_value", F.when(F.col("orders") > 0, F.col("revenue") / F.col("orders")))
    .orderBy("month_start")
)

country = (
    sales.groupBy("country")
    .agg(
        F.sum("revenue").alias("revenue"),
        F.sum("quantity").alias("units"),
        F.countDistinct("invoice_id").alias("orders"),
        F.countDistinct("customer_id_num").alias("customers")
    )
    .withColumn("average_order_value", F.when(F.col("orders") > 0, F.col("revenue") / F.col("orders")))
    .withColumn("revenue_share", F.col("revenue") / F.sum("revenue").over(Window.partitionBy()))
    .orderBy(F.desc("revenue"))
)

product = (
    sales.groupBy("stock_code", "description")
    .agg(
        F.sum("revenue").alias("revenue"),
        F.sum("quantity").alias("units"),
        F.countDistinct("invoice_id").alias("orders"),
        F.countDistinct("customer_id_num").alias("customers")
    )
    .withColumn("average_unit_price", F.when(F.col("units") > 0, F.col("revenue") / F.col("units")))
    .orderBy(F.desc("revenue"))
)

for df, name in [
    (daily_country, gold_daily_table),
    (monthly, gold_monthly_table),
    (country, gold_country_table),
    (product, gold_product_table),
]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(name)

print("Gold aggregate tables written")

## 2. Customer RFM features

Recency is the number of days since a customer's last valid purchase. Frequency is the number of distinct invoices. Monetary is the customer's total revenue. The model is a descriptive segmentation tool, not a claim that a customer will behave a certain way in the future.

In [ ]:
max_sale_date = sales.select(F.max("sale_date").alias("max_date")).first()["max_date"]

customer_rfm = (
    sales.filter(F.col("customer_id_num").isNotNull())
    .groupBy("customer_id_num")
    .agg(
        F.datediff(F.lit(max_sale_date), F.max("sale_date")).alias("recency_days"),
        F.countDistinct("invoice_id").alias("frequency_orders"),
        F.sum("revenue").alias("monetary_revenue"),
        F.countDistinct("stock_code").alias("distinct_products"),
        F.min("sale_date").alias("first_purchase_date"),
        F.max("sale_date").alias("last_purchase_date")
    )
    .filter((F.col("monetary_revenue") > 0) & (F.col("frequency_orders") > 0))
)

customer_rfm.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(gold_customer_table)
print(f"Customer RFM rows: {customer_rfm.count()}")
display(customer_rfm.orderBy(F.desc("monetary_revenue")).limit(10))

## 3. Exploratory data analysis

These views support the dashboard. They answer three business questions from the project brief:

1. How does revenue change over time?
2. Which countries and products contribute the most revenue?
3. Which customer groups should receive different retention or engagement actions?

In [ ]:
kpis = sales.agg(
    F.sum("revenue").alias("total_revenue"),
    F.sum("quantity").alias("total_units"),
    F.countDistinct("invoice_id").alias("total_orders"),
    F.countDistinct("customer_id_num").alias("known_customers"),
    F.countDistinct("stock_code").alias("active_products"),
    F.min("sale_date").alias("start_date"),
    F.max("sale_date").alias("end_date")
)
display(kpis)

display(monthly.orderBy("month_start"))
display(country.limit(15))
display(product.limit(15))

# Dashboard-ready chart data. Create Databricks visualisations from these results:
# line chart: monthly revenue; bar chart: revenue by country; bar chart: top products.

## 4. RFM customer segmentation model

The model standardises recency, frequency, and monetary value before fitting KMeans. The notebook evaluates several candidate values of `k` using silhouette score. Higher silhouette indicates better separation between clusters. The within-cluster sum of squared errors is also stored to show the elbow trade-off as `k` increases.

In [ ]:
feature_cols = ["recency_days", "frequency_orders", "monetary_revenue"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette", distanceMeasure="squaredEuclidean")

prepared = scaler.fit(assembler.transform(customer_rfm)).transform(assembler.transform(customer_rfm)).select("customer_id_num", *feature_cols, "features")
model_rows = []
for k in range(2, 7):
    model = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction")
    fitted = model.fit(prepared)
    scored = fitted.transform(prepared)
    silhouette = evaluator.evaluate(scored)
    model_rows.append((k, float(silhouette), float(fitted.summary.trainingCost), int(scored.select("customer_id_num").count())))

eval_schema = T.StructType([
    T.StructField("k", T.IntegerType(), False),
    T.StructField("silhouette", T.DoubleType(), False),
    T.StructField("within_set_sum_squared_error", T.DoubleType(), False),
    T.StructField("customers_evaluated", T.IntegerType(), False),
])
model_eval = spark.createDataFrame(model_rows, eval_schema).orderBy(F.desc("silhouette"))
model_eval.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(model_eval_table)
display(model_eval)

best_k = model_eval.first()["k"]
final_kmeans = KMeans(k=int(best_k), seed=42, featuresCol="features", predictionCol="cluster_id")
final_model = final_kmeans.fit(prepared)
segments = (
    final_model.transform(prepared)
    .withColumn("segment_label", F.concat(F.lit("RFM_Cluster_"), F.col("cluster_id").cast("string")))
    .select("customer_id_num", *feature_cols, "cluster_id", "segment_label")
)
segments.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(model_table)

segment_summary = (
    segments.groupBy("cluster_id", "segment_label")
    .agg(
        F.count("customer_id_num").alias("customers"),
        F.avg("recency_days").alias("avg_recency_days"),
        F.avg("frequency_orders").alias("avg_frequency_orders"),
        F.avg("monetary_revenue").alias("avg_monetary_revenue"),
        F.sum("monetary_revenue").alias("segment_revenue")
    )
    .withColumn("segment_revenue_share", F.col("segment_revenue") / F.sum("segment_revenue").over(Window.partitionBy()))
    .orderBy("cluster_id")
)
display(segment_summary)

## 5. Evaluation summary

The model is accepted only when the selected clustering run has a valid silhouette score, no empty cluster, and a segment summary that can be explained in business terms. The following cell creates a compact evaluation table for the report and dashboard.

In [ ]:
selected_eval = model_eval.filter(F.col("k") == int(best_k)).withColumn("metric", F.lit("silhouette_and_training_cost"))
cluster_sizes = segments.groupBy("cluster_id").count()
evaluation_summary = (
    selected_eval
    .crossJoin(cluster_sizes.agg(F.min("count").alias("smallest_cluster"), F.max("count").alias("largest_cluster")))
    .withColumn("all_clusters_non_empty", F.col("smallest_cluster") > 0)
    .withColumn("model_accepted", (F.col("silhouette") > 0) & F.col("all_clusters_non_empty"))
)
display(evaluation_summary)

# Optional model sanity checks for the final report.
print(f"Selected k: {best_k}")
print(f"Silhouette score: {model_eval.first()['silhouette']:.4f}")
print("Model acceptance rule: silhouette > 0 and every cluster contains at least one customer.")

## 6. Dashboard build instructions

In Databricks, create a dashboard from the result tables or SQL queries below. Keep the dashboard to three main charts and a KPI strip so that it answers the project questions clearly.

**KPI strip:** total revenue, total orders, known customers, active products, date range.

**Chart 1: Monthly revenue:** use `retail_project_gold_monthly`, with `month_start` on the x-axis and `revenue` as the value.

**Chart 2: Country revenue:** use `retail_project_gold_country`, sorted by `revenue`, limited to the top 10 countries.

**Chart 3: Customer segments:** use the `segment_summary` result, showing customers and segment revenue share. Add a table below it with average recency, frequency, and monetary value so the segment labels remain interpretable.

Add a dashboard note: revenue excludes cancelled invoices and invalid rows; customer IDs are pseudonymous analytical keys; results are based on the source period in the dataset.